In [0]:
%sql
CREATE OR REPLACE VIEW datalakehouse.gold.vw_salario AS
WITH base AS (
    SELECT
        data_referencia as data,
        valor AS salario_anual,
        ano,
        num_mes as mes,
        moeda
    FROM datalakehouse.silver.tbl_salario_serie_1619
),

-- Preenche os meses que não têm salário, usando o último valor válido do ano
preenchido AS (
    SELECT
        data,
        ano,
        mes,
        LAST_VALUE(salario_anual, TRUE)
            OVER (PARTITION BY ano ORDER BY mes
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS salario_nominal
    FROM base
),

-- Calculo da variação anual apenas no primeiro mês do novo valor
variacao AS (
    SELECT
        data,
        ano,
        mes,
        salario_nominal,
        CASE
            WHEN salario_nominal = LAG(salario_nominal) OVER (ORDER BY data)
            THEN 0
            ELSE ROUND(
                try_divide(
                    salario_nominal,
                    LAG(salario_nominal) OVER (ORDER BY data)
                ) - 1,
                6
            )
        END AS salario_var_anual
    FROM preenchido
)

SELECT *
FROM variacao
ORDER BY data;
